In [ ]:
!pip install -q "git+https://github.com/djouallah/duckrun.git@main" --upgrade
import notebookutils
notebookutils.session.restartPython()


In [ ]:
# Run the coffee dbt project (staged into this lakehouse's Files/coffee) on Fabric compute.
import os, requests, notebookutils
ctx = notebookutils.runtime.context
ws = ctx.get("currentWorkspaceId") or ctx.get("workspaceId")
token = notebookutils.credentials.getToken("pbi")
lakehouses = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{ws}/lakehouses",
    headers={"Authorization": f"Bearer {token}"}).json()["value"]
lh = next(l["id"] for l in lakehouses if l["displayName"] == "deploy_demo")
root = f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"

os.environ["WAREHOUSE_PATH"] = f"{root}/Tables"   # coffee's profiles.yml reads this
os.environ["DBT_SCHEMA"] = "dbo"

proj = "/tmp/coffee"
notebookutils.fs.cp(f"{root}/Files/coffee", f"file:{proj}", True)
os.chdir(proj)

from dbt.cli.main import dbtRunner
res = dbtRunner().invoke(["build", "--project-dir", proj, "--profiles-dir", proj])
if not res.success:
    raise SystemExit("dbt build failed")
print("coffee built on Fabric ✅")
